# Vehicle & License Plate Detection with YOLOv8

Detects **13 classes** of Indonesian road users in a single image: cars, motorbikes, pickups, city minibuses (*angkot*), pedicabs (*becak*), carts, bicycles, buses and trucks in two sizes, plus **license plates**.
The approach is transfer learning from COCO-pretrained YOLOv8, with a comparison of YOLOv8s against the lighter YOLOv8n.

**Dataset:** [Roboflow Universe: deteksi-kendaraan v5](https://universe.roboflow.com/irfan-maulana/deteksi-kendaraan/dataset/5) (CC BY 4.0). 1,347 train / 440 val / 212 test images.

In [9]:
import os
import glob
import random
import yaml
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Set these before importing torch / initialising CUDA.
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torchvision
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

# Make CUDA/cuDNN operations deterministic.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# warn_only: keep running even if an op has no deterministic implementation.
torch.use_deterministic_algorithms(True, warn_only=True)

# Use the GPU if available, otherwise fall back to CPU.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Seed:        {RANDOM_STATE}")
print(f"Torch:       {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()} (built with {torch.version.cuda})")
print(f"Device used: {DEVICE}")

Seed:        42
Torch:       2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA:        True (built with 12.8)
Device used: cuda


In [10]:
# Build ./data first with: python prepare_dataset.py <roboflow export zip>  (see README)
DATA_PATH = os.path.abspath("data/data.yaml")

with open(DATA_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

# Point the splits at folders relative to data.yaml
data_config['path'] = os.path.dirname(DATA_PATH)
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

with open(DATA_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml updated!")
print(data_config)

data.yaml updated!
{'names': ['Angkot', 'Becak', 'Bus Besar', 'Bus Sedang', 'Gerobag', 'Gerobak', 'Mobil', 'Motor', 'Pick Up', 'Plat', 'Sepeda', 'Truk Besar', 'Truk Sedang'], 'nc': 13, 'roboflow': {'license': 'CC BY 4.0', 'project': 'deteksi-kendaraan', 'url': 'https://universe.roboflow.com/irfan-maulana/deteksi-kendaraan/dataset/5', 'version': 5, 'workspace': 'irfan-maulana'}, 'test': 'test/images', 'train': 'train/images', 'val': 'valid/images'}


In [11]:
# ===== 1. DATASET & PREPROCESSING =====
print("=== Dataset Configuration ===")
print(f"Classes ({data_config['nc']}): {data_config['names']}")

for split in ['train', 'val', 'test']:
    img_dir = data_config[split]
    if not os.path.isabs(img_dir):
        img_dir = os.path.join(os.path.dirname(DATA_PATH), img_dir)
    n = len(glob.glob(os.path.join(img_dir, '*.*')))
    print(f"{split}: {n} images")

print("\n=== Preprocessing ===")
print("- Data source: Roboflow")
print("- Format: YOLOv8 (images + txt labels)")
print("- Data split: train / valid / test (from Roboflow)")
print("- Image resize: 640x640 (during training)")
print("- Augmentation: YOLOv8 default (mosaic, flip, HSV augmentation)")

=== Dataset Configuration ===
Classes (13): ['Angkot', 'Becak', 'Bus Besar', 'Bus Sedang', 'Gerobag', 'Gerobak', 'Mobil', 'Motor', 'Pick Up', 'Plat', 'Sepeda', 'Truk Besar', 'Truk Sedang']
train: 1347 images
val: 440 images
test: 212 images

=== Preprocessing ===
- Data source: Roboflow
- Format: YOLOv8 (images + txt labels)
- Data split: train / valid / test (from Roboflow)
- Image resize: 640x640 (during training)
- Augmentation: YOLOv8 default (mosaic, flip, HSV augmentation)


## 1. Dataset & Preprocessing

The dataset is in YOLO format (images + `.txt` bounding-box labels) and comes pre-split into **train / validation / test**. The training set teaches the model, the validation set drives early stopping, and the test set is only used for the final evaluation.

About 2,000 images is small for object detection, so the model starts from **COCO-pretrained weights** (transfer learning). Preprocessing, augmentation and model size all matter for avoiding overfitting.

**Preprocessing**
- Every image is resized to **640×640** so batches are uniform.
- Ultralytics' built-in augmentation (mosaic, horizontal flip, HSV jitter) simulates different lighting, positions and object layouts.

**Why this matters here:** vehicles and plates appear at very different sizes and angles. License plates are much smaller than the vehicles, so consistent resizing and augmentation help the model learn features that hold up.

In [12]:
# ===== 2. MODEL =====
model = YOLO('yolov8s.pt')  # YOLOv8 Small, pretrained on COCO

print("=== Model Architecture ===")
print("- Model: YOLOv8s (Small)")
print("- Pretrained: COCO dataset")
print("- Task: Object Detection")
print(f"- Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

=== Model Architecture ===
- Model: YOLOv8s (Small)
- Pretrained: COCO dataset
- Task: Object Detection
- Parameters: 11,166,560


In [13]:
results = model.train(
    data=DATA_PATH,
    epochs=30,
    imgsz=640,
    batch=8,
    name='vehicle_detection',
    patience=10,
    device=DEVICE,
    workers=0,
    seed=RANDOM_STATE,
    deterministic=True,
    exist_ok=True,
)

RESULTS_DIR = str(results.save_dir)

## 2. Model & Training Setup

The main model is **YOLOv8s** (~11.1M parameters) pretrained on COCO. It balances detection accuracy against compute, which matters because training ran on a 6 GB laptop GPU. COCO pretraining already covers general concepts like cars, people and object shapes, so the model converges much faster on a small dataset.

**Training configuration**
- 30 epochs, image size 640×640, batch size 8
- early stopping with patience 10
- seed 42, deterministic mode on (reproducible runs)

A batch size of 8 fits in GPU memory while keeping gradients stable. Early stopping ends training once validation performance stops improving.

In [14]:
# ===== 3. TRAINING & VALIDATION RESULTS =====
results_img = Image.open(os.path.join(RESULTS_DIR, 'results.png'))
plt.figure(figsize=(15, 8))
plt.imshow(results_img)
plt.axis('off')
plt.title('Training & Validation Results')
plt.show()

## 3. Training Curves

![Training curves](images/training_curves.png)

- **Training losses** (`box`, `cls`, `dfl`) fall steadily: the model gets better at localising boxes, classifying objects and refining box edges.
- **Validation losses** follow the same downward trend with no divergence, so there is no sign of severe overfitting.
- **Precision** fluctuates but trends upward, ending around 0.77–0.80. **Recall** rises steadily as the model detects more objects.
- **mAP50** and **mAP50-95** increase consistently across training.

The remaining noise in precision and recall is expected on a small dataset, where each batch varies a lot.

In [15]:
# ===== 4. EVALUATION =====
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, 'weights', 'best.pt')
best_model = YOLO(BEST_MODEL_PATH)

metrics = best_model.val(
    data=DATA_PATH,
    split='test',
    seed=RANDOM_STATE,
    deterministic=True,
    workers=0,
    name='vehicle_detection_yolov8s_test_eval',
    exist_ok=True,
)

TEST_EVAL_DIR = str(metrics.save_dir)

print("=== Test Set Evaluation: YOLOv8s ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"Best model path: {BEST_MODEL_PATH}")

# Prefer confusion matrix from the test evaluation folder. If it is not available, use the training folder version.
cm_path = os.path.join(TEST_EVAL_DIR, 'confusion_matrix.png')
if not os.path.exists(cm_path):
    cm_path = os.path.join(RESULTS_DIR, 'confusion_matrix.png')

if os.path.exists(cm_path):
    cm_img = Image.open(cm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(cm_img)
    plt.axis('off')
    plt.title('Confusion Matrix - YOLOv8s')
    plt.show()
else:
    print("Confusion matrix image was not found. Check the validation output folder.")


Ultralytics 8.4.49  Python-3.10.11 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 11,130,615 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 1.11.9 ms, read: 7.41.7 MB/s, size: 52.8 KB)
val: Scanning data\test\labels... 212 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 212/212 948.1it/s 0.2s0.0s
val: New cache created: data\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 3.5it/s 4.0s0.3s
                   all        212       1432      0.793      0.765      0.814      0.574
                Angkot          8          8      0.814      0.551      0.688       0.56
                 Becak         19         22      0.808      0.818       0.82      0.523
             Bus Besar          8          9      0.725      0.778      0.727      0.538
            Bus Sedang         29         31      0.801      0.774      0.789      0

### Test-set results (YOLOv8s)

| Metric | Value |
|---|---|
| mAP50 | **0.814** |
| mAP50-95 | 0.574 |
| Precision | 0.793 |
| Recall | 0.765 |

- **Cars (Mobil) and motorbikes (Motor)** are detected best (mAP50 ≈ 0.95 and 0.93), since they have the most training examples.
- **License plates (Plat)** are the hardest class (mAP50 0.73, mAP50-95 0.40). Plates are small, so tight box localisation is difficult.
- Rare classes (large buses, large trucks, bicycles) have fewer than 12 test images each, so their scores are noisy.

![Confusion matrix](images/confusion_matrix_yolov8s.png)

## 4. Model Comparison: YOLOv8s vs YOLOv8n

Is the larger model worth it? **YOLOv8n** (nano, ~3M parameters) is trained with exactly the same data, image size, epochs, batch size and seed, so the comparison is fair. Nano is ~3.7× smaller and roughly twice as fast, which matters for deployment on edge devices such as a roadside camera.

In [16]:
# ===== 5. MODEL COMPARISON =====
# Comparison model: YOLOv8n (Nano)
# Goal: compare accuracy vs speed/model size against the main YOLOv8s model.


EPOCHS = 30
IMG_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 10


def count_parameters(yolo_model):
    """Count total parameters in an Ultralytics YOLO model."""
    return sum(p.numel() for p in yolo_model.model.parameters())


def safe_metric(value):
    """Convert metric values safely to float."""
    try:
        return float(value)
    except Exception:
        return np.nan


def build_comparison_row(model_name, role, model_path, metrics_obj, notes):
    """Create one row of evaluation results for model comparison."""
    loaded_model = YOLO(model_path)
    speed = getattr(metrics_obj, 'speed', {}) or {}

    return {
        'Model': model_name,
        'Role': role,
        'Parameters': count_parameters(loaded_model),
        'mAP50': round(safe_metric(metrics_obj.box.map50), 4),
        'mAP50-95': round(safe_metric(metrics_obj.box.map), 4),
        'Precision': round(safe_metric(metrics_obj.box.mp), 4),
        'Recall': round(safe_metric(metrics_obj.box.mr), 4),
        'Inference (ms/img)': round(safe_metric(speed.get('inference', np.nan)), 3),
        'Best Weights Path': model_path,
        'Notes': notes,
    }


comparison_rows = []

# Add the already-trained main model: YOLOv8s
comparison_rows.append(
    build_comparison_row(
        model_name='YOLOv8s',
        role='Main model',
        model_path=BEST_MODEL_PATH,
        metrics_obj=metrics,
        notes='Higher capacity, better candidate if accuracy is the priority.'
    )
)

# Train the lighter comparison model: YOLOv8n
print("=== Training comparison model: YOLOv8n ===")
yolov8n_model = YOLO('yolov8n.pt')

nano_results = yolov8n_model.train(
    data=DATA_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name='vehicle_detection_yolov8n',
    patience=PATIENCE,
    device=DEVICE,
    workers=0,
    seed=RANDOM_STATE,
    deterministic=True,
    exist_ok=True,
)

NANO_RESULTS_DIR = str(nano_results.save_dir)
NANO_BEST_MODEL_PATH = os.path.join(NANO_RESULTS_DIR, 'weights', 'best.pt')

# Evaluate YOLOv8n on the same test split
nano_best_model = YOLO(NANO_BEST_MODEL_PATH)
nano_metrics = nano_best_model.val(
    data=DATA_PATH,
    split='test',
    seed=RANDOM_STATE,
    deterministic=True,
    workers=0,
    name='vehicle_detection_yolov8n_test_eval',
    exist_ok=True,
)

comparison_rows.append(
    build_comparison_row(
        model_name='YOLOv8n',
        role='Comparison model',
        model_path=NANO_BEST_MODEL_PATH,
        metrics_obj=nano_metrics,
        notes='Lighter and faster, useful baseline for small/medium dataset and deployment.'
    )
)

# Create comparison table
model_comparison = pd.DataFrame(comparison_rows)
model_comparison['Rank by mAP50-95'] = model_comparison['mAP50-95'].rank(ascending=False, method='min').astype(int)
model_comparison = model_comparison.sort_values(by=['mAP50-95', 'mAP50'], ascending=False).reset_index(drop=True)

print("=== Model Comparison Table ===")
display(model_comparison)

# Save the comparison table
comparison_csv_path = 'model_comparison_results.csv'
model_comparison.to_csv(comparison_csv_path, index=False)
print(f"Saved comparison table as: {comparison_csv_path}")

# Accuracy comparison
plt.figure(figsize=(7, 5))
plt.bar(model_comparison['Model'], model_comparison['mAP50-95'])
plt.title('Model Comparison by mAP50-95')
plt.xlabel('Model')
plt.ylabel('mAP50-95')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_map50_95.png', dpi=300, bbox_inches='tight')
plt.show()

# Speed comparison
plt.figure(figsize=(7, 5))
plt.bar(model_comparison['Model'], model_comparison['Inference (ms/img)'])
plt.title('Model Comparison by Inference Speed')
plt.xlabel('Model')
plt.ylabel('Inference Time (ms/img)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_inference_speed.png', dpi=300, bbox_inches='tight')
plt.show()

# Recommended model for deployment based on stricter accuracy metric.
best_row = model_comparison.iloc[0]
DEPLOY_MODEL_NAME = best_row['Model']
DEPLOY_MODEL_PATH = best_row['Best Weights Path']

summary_text = f"""
=== Model Comparison Conclusion ===
Best model by mAP50-95: {DEPLOY_MODEL_NAME}
Best weights path: {DEPLOY_MODEL_PATH}

Interpretation:
- YOLOv8s is the main model and is expected to perform better when accuracy is the priority.
- YOLOv8n is the comparison model and is useful when faster inference and lighter deployment are needed.
- Final choice: the model with the best mAP50-95; precision, recall, inference speed and parameter count describe the trade-off.
"""

print(summary_text)

with open('model_comparison_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print("Saved summary as: model_comparison_summary.txt")


,Model,Role,Parameters,mAP50,mAP50-95,Precision,Recall,Inference (ms/img),Best Weights Path,Notes,Rank by mAP50-95
0,YOLOv8s,Main model,11140631,0.8140,0.5744,0.7932,0.7649,10.527,runs\detect\...,"Higher capacity, better candidate if accuracy ...",1
1,YOLOv8n,Comparison model,3013383,0.7612,0.5326,0.7483,0.6779,5.140,runs\detect\...,"Lighter and faster, useful baseline for small/...",2


Saved comparison table as: model_comparison_results.csv



=== Model Comparison Conclusion ===
Best model by mAP50-95: YOLOv8s
Best weights path: runs\detect\vehicle_detection\weights\best.pt

Interpretation:
- YOLOv8s is the main model and is expected to perform better when accuracy is the priority.
- YOLOv8n is the comparison model and is useful when faster inference and lighter deployment are needed.
- Final choice: the model with the best mAP50-95; precision, recall, inference speed and parameter count describe the trade-off.

Saved summary as: model_comparison_summary.txt


In [17]:
test_img_dir = data_config['test']
if not os.path.isabs(test_img_dir):
    test_img_dir = os.path.join(os.path.dirname(DATA_PATH), test_img_dir)

pred_results = best_model.predict(
    source=test_img_dir,
    save=True,
    conf=0.25,
    name='test_predictions_yolov8s',
    seed=RANDOM_STATE,
    exist_ok=True,
)

pred_dir = str(pred_results[0].save_dir)
pred_images = sorted(glob.glob(os.path.join(pred_dir, '*.jpg')))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), pred_images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('Sample Predictions on Test Set')
plt.tight_layout()
plt.show()

## 5. Conclusions

| Model | Params | mAP50 | mAP50-95 | Precision | Recall | Inference (ms/img) |
|---|---|---|---|---|---|---|
| **YOLOv8s** | 11.1M | **0.814** | **0.574** | **0.793** | **0.765** | 10.5 |
| YOLOv8n | 3.0M | 0.761 | 0.533 | 0.748 | 0.678 | **5.1** |

- **YOLOv8s is the final model.** It is better on every accuracy metric, and recall improves the most (+8.7 pts).
- **YOLOv8n is the choice when speed matters.** It runs about twice as fast with 3.7× fewer parameters and costs ~4 pts of mAP50-95.
- Neither model shows overfitting: training and validation losses fall together.
- **License plates are the weakest class.** The next step would be a dedicated plate detector followed by OCR, or training at a higher resolution for small objects.